# Extension 5 -- Strong First-Birth AIPW Analysis

**Research question.** Does the adjusted public-private Cesarean gap remain among first births, where prior Cesarean history is impossible by construction?

**Why this matters.** Extension 1's quantitative bias analysis asks "how big would an unmeasured confounder like prior Cesarean history need to be to explain the gap?" This extension answers a complementary question directly: restrict to women having their *first* birth, for whom prior Cesarean cannot exist as a confounder at all -- not because it's assumed away, but because it is logically impossible. If the gap survives here, that specific confounding concern is neutralized by construction for this subgroup.

**Relationship to Notebook 08.** `notebooks/v2/08_risk_stratified_aipw.ipynb` already stratifies the *primary* model's first-birth records into lower/elevated-risk strata. This extension is different: it builds a **dedicated** first-birth-only AIPW model with its **own** confounder set (swapping out the now-constant `birth_order` for `age_at_first_birth`, which is exact only for this subgroup) and its own cross-fitting -- not a post-hoc stratification of the primary model's already-fitted nuisances.

**What this does and does not remove.** First-birth restriction eliminates *prior Cesarean history* as a confounder by construction. It does **not** remove other unmeasured first-birth-specific obstetric indications (e.g. fetal compromise, labor dystocia, cephalopelvic disproportion) that NFHS-5 also doesn't capture -- this notebook does not claim otherwise, and does not claim to be the first low-risk/nulliparous sector analysis in India (per the handoff's explicit QA rule).

**Not run here.** Written but not executed -- this sandbox has no NFHS data. Run it locally where `data/processed/df_model_v2.parquet` already exists.

## 1. Imports, shared config/utils

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path("../shared").resolve()))
import config
import utils

pd.set_option("display.max_columns", None)

OUTPUTS_DIR = config.EXT05_OUTPUTS_DIR
RANDOM_STATE = config.RANDOM_STATE
N_FOLDS = config.N_FOLDS
N_BOOTSTRAP = config.N_BOOTSTRAP

## 2. First-birth cohort construction (handoff step 37)

`birth_order == 1` is used to identify first births -- **not** `birth_index`, which is a row-position bookkeeping field that can diverge from true birth order (see `notebooks/v2/01`'s documentation of exactly this distinction). Records missing `age_at_first_birth` are excluded from the modeling set rather than backfilled with `maternal_age` (handoff step 38 explicitly forbids that substitution), and the exclusion count is reported, not hidden.

In [ ]:
analytic, n_loaded, n_other = utils.load_public_private_analytic(
    config.V2_PROCESSED_DATA_PATH,
    expected_total_rows=config.EXPECTED_V2_ROW_COUNT,
    expected_analytic_rows=config.EXPECTED_ANALYTIC_ROW_COUNT,
)
print(f"Full analytic cohort (public + private): {len(analytic)}")

first_birth = analytic[analytic["birth_order"] == 1].copy()
print(f"First-birth records (birth_order == 1): {len(first_birth)} "
      f"({len(first_birth) / len(analytic) * 100:.2f}% of the analytic cohort)")

n_missing_age = first_birth["age_at_first_birth"].isna().sum()
first_birth_modeling = first_birth[first_birth["age_at_first_birth"].notna()].copy()
print(f"Excluded for missing age_at_first_birth (not substituted with maternal_age): {n_missing_age} "
      f"({n_missing_age / len(first_birth) * 100:.2f}% of first births)")
print(f"First-birth modeling cohort: {len(first_birth_modeling)}")

In [ ]:
def sector_prevalence_table(df):
    rows = []
    for facility in ["public", "private"]:
        sub = df[df["facility_type"] == facility]
        rows.append({
            "facility_type": facility, "n": len(sub),
            "raw_csection_prevalence_pct": sub["csection"].mean() * 100 if len(sub) else np.nan,
            "weighted_csection_prevalence_pct": (
                np.average(sub["csection"], weights=sub["sample_weight_normalized"]) * 100 if len(sub) else np.nan
            ),
        })
    return pd.DataFrame(rows)


cohort_summary = sector_prevalence_table(first_birth_modeling)
cohort_summary.insert(0, "cohort", "first_birth_modeling")
cohort_summary.insert(1, "n_total", len(first_birth_modeling))
cohort_summary.insert(2, "n_excluded_missing_age_at_first_birth", n_missing_age)

cohort_summary.to_csv(OUTPUTS_DIR / "first_birth_cohort_summary.csv", index=False)
print("Saved:", OUTPUTS_DIR / "first_birth_cohort_summary.csv")
cohort_summary

## 3. First-birth-only confounder set (handoff step 39)

`birth_order` is constant (== 1) within this cohort and is dropped rather than kept as a degenerate predictor. `age_at_first_birth` (`v212`) replaces it -- unlike `maternal_age`, this variable is exact age-at-the-analyzed-birth *by construction* for first-birth records (documented in `notebooks/v2/06`), so it is safe to use here even though `maternal_age` remains excluded project-wide.

In [ ]:
FIRST_BIRTH_NUMERIC = ["age_at_first_birth", "wealth_index", "education_years"]
FIRST_BIRTH_CATEGORICAL = ["residence", "religion", "social_group", "twin_order", "state"]
FIRST_BIRTH_CONFOUNDERS = FIRST_BIRTH_NUMERIC + FIRST_BIRTH_CATEGORICAL

assert "birth_order" not in FIRST_BIRTH_CONFOUNDERS, \
    "birth_order is constant within the first-birth cohort and must not remain as a predictor."
assert first_birth_modeling["birth_order"].nunique() == 1, \
    "birth_order should be constant (==1) in this cohort -- investigate before proceeding."

print(f"First-birth confounder set ({len(FIRST_BIRTH_CONFOUNDERS)}): {FIRST_BIRTH_CONFOUNDERS}")
print("Confirmed: birth_order is constant in this cohort and excluded as a predictor.")

## 4. Overlap and covariate-balance diagnostics within the first-birth cohort (handoff step 40)

In [ ]:
W_diag = first_birth_modeling[FIRST_BIRTH_CONFOUNDERS].copy()
for col in FIRST_BIRTH_CATEGORICAL:
    W_diag[col] = W_diag[col].astype("string").fillna("Missing").astype(str)
for col in FIRST_BIRTH_NUMERIC:
    W_diag[col] = W_diag[col].fillna(W_diag[col].median())

exposure_fb = first_birth_modeling["exposure"].to_numpy()
weight_fb = first_birth_modeling["sample_weight_normalized"].to_numpy()

prop_pipeline = utils.make_propensity_pipeline(FIRST_BIRTH_NUMERIC, FIRST_BIRTH_CATEGORICAL, RANDOM_STATE)
prop_pipeline.fit(W_diag, exposure_fb, clf__sample_weight=weight_fb)
e_hat_diag = np.clip(prop_pipeline.predict_proba(W_diag)[:, 1], config.CLIP_EPS, 1 - config.CLIP_EPS)

marginal_private_fb = np.average(exposure_fb, weights=weight_fb)
stabilized_ipw_fb = np.where(
    exposure_fb == 1, marginal_private_fb / e_hat_diag, (1 - marginal_private_fb) / (1 - e_hat_diag)
)
combined_weight_fb = weight_fb * stabilized_ipw_fb

balance_dummies = pd.concat([
    W_diag[FIRST_BIRTH_NUMERIC],
    pd.get_dummies(W_diag[FIRST_BIRTH_CATEGORICAL], columns=FIRST_BIRTH_CATEGORICAL, prefix=FIRST_BIRTH_CATEGORICAL),
], axis=1)

exposure_mask_fb = exposure_fb == 1
unweighted_ones_fb = np.ones(len(first_birth_modeling))

balance_rows = []
for col in balance_dummies.columns:
    values = balance_dummies[col].to_numpy(dtype=float)
    smd_before = utils.smd(values[exposure_mask_fb], unweighted_ones_fb[exposure_mask_fb],
                            values[~exposure_mask_fb], unweighted_ones_fb[~exposure_mask_fb])
    smd_after = utils.smd(values[exposure_mask_fb], combined_weight_fb[exposure_mask_fb],
                           values[~exposure_mask_fb], combined_weight_fb[~exposure_mask_fb])
    balance_rows.append({"covariate": col, "smd_before": smd_before, "smd_after": smd_after})

first_birth_overlap_balance = pd.DataFrame(balance_rows).sort_values("smd_before", key=np.abs, ascending=False)
first_birth_overlap_balance["flag_after_weighting"] = first_birth_overlap_balance["smd_after"].abs() > 0.1
first_birth_overlap_balance.to_csv(OUTPUTS_DIR / "first_birth_overlap_balance.csv", index=False)

n_flagged = int(first_birth_overlap_balance["flag_after_weighting"].sum())
ess_private = utils.effective_sample_size(combined_weight_fb[exposure_mask_fb])
ess_public = utils.effective_sample_size(combined_weight_fb[~exposure_mask_fb])
print(f"Saved: {OUTPUTS_DIR / 'first_birth_overlap_balance.csv'}")
print(f"Covariates with |SMD| > 0.1 after weighting: {n_flagged} of {len(first_birth_overlap_balance)}")
print(f"Effective sample size after weighting -- private: {ess_private:.0f}, public: {ess_public:.0f}")
print(f"Common support (propensity) range: [{e_hat_diag.min():.4f}, {e_hat_diag.max():.4f}]")

## 5. Respondent-grouped cross-fitted AIPW (handoff step 41)

Same architecture as `notebooks/v2/07_aipw_primary_analysis.ipynb`, applied to the first-birth-only cohort with the first-birth-only confounder set, via `shared/utils.run_cross_fitted_aipw`.

In [ ]:
first_birth_result = utils.run_cross_fitted_aipw(
    first_birth_modeling, FIRST_BIRTH_CONFOUNDERS, FIRST_BIRTH_NUMERIC, FIRST_BIRTH_CATEGORICAL,
    n_folds=N_FOLDS, random_state=RANDOM_STATE, n_bootstrap=N_BOOTSTRAP, bootstrap_seed=config.BOOTSTRAP_SEED,
)

print(f"First-birth adjusted private risk: {first_birth_result['r1'] * 100:.2f}%")
print(f"First-birth adjusted public risk: {first_birth_result['r0'] * 100:.2f}%")
print(f"First-birth adjusted RD: {first_birth_result['risk_difference'] * 100:.2f} pp "
      f"[{first_birth_result['rd_ci'][0] * 100:.2f}, {first_birth_result['rd_ci'][1] * 100:.2f}]")
print(f"First-birth adjusted RR: {first_birth_result['risk_ratio']:.3f} "
      f"[{first_birth_result['rr_ci'][0]:.3f}, {first_birth_result['rr_ci'][1]:.3f}]")

## 6. Save AIPW summary and bootstrap results (handoff step 42)

In [ ]:
first_birth_aipw_summary = pd.DataFrame([{
    "r1_hat_private_risk_pct": first_birth_result["r1"] * 100,
    "r0_hat_public_risk_pct": first_birth_result["r0"] * 100,
    "risk_difference_pct": first_birth_result["risk_difference"] * 100,
    "risk_difference_ci_low_pct": first_birth_result["rd_ci"][0] * 100,
    "risk_difference_ci_high_pct": first_birth_result["rd_ci"][1] * 100,
    "risk_ratio": first_birth_result["risk_ratio"],
    "risk_ratio_ci_low": first_birth_result["rr_ci"][0],
    "risk_ratio_ci_high": first_birth_result["rr_ci"][1],
    "n_analytic": first_birth_result["n"],
    "n_private": int((first_birth_modeling['exposure'] == 1).sum()),
    "n_public": int((first_birth_modeling['exposure'] == 0).sum()),
}])
first_birth_aipw_summary.to_csv(OUTPUTS_DIR / "first_birth_aipw_summary.csv", index=False)
print("Saved:", OUTPUTS_DIR / "first_birth_aipw_summary.csv")

first_birth_bootstrap_summary = pd.DataFrame({
    "replicate": np.arange(N_BOOTSTRAP),
    "risk_difference_pct": first_birth_result["rd_reps"] * 100,
    "risk_ratio": first_birth_result["rr_reps"],
})
first_birth_bootstrap_summary.to_csv(OUTPUTS_DIR / "first_birth_bootstrap_summary.csv", index=False)
print("Saved:", OUTPUTS_DIR / "first_birth_bootstrap_summary.csv")
first_birth_aipw_summary

## 7. Comparison with the full-cohort primary estimate (handoff step 43)

**Why this is a side-by-side comparison, not a formal joint confidence interval for the difference.**
First-birth women are a *subset* of the full analytic cohort, not an independent sample (unlike Extension 3's
NFHS-4-vs-NFHS-5 comparison, where the two samples genuinely don't overlap). Pairing this notebook's bootstrap
replicates index-by-index against the frozen primary model's replicates (`outputs/tables/b4_aipw_bootstrap_results.csv`)
the way Extension 3 does for NFHS-4/NFHS-5 would silently assume independence that does not hold here, understating
the true correlation between the two estimates and producing an invalid interval. A statistically valid joint CI
would require re-resampling the *same* PSU draws for both models inside one bootstrap loop using saved row-level
nuisance predictions from both fits -- which needs the full-cohort's row-level nuisance file (not guaranteed to
exist locally) and roughly doubles the computational cost. That is flagged here as a possible future enhancement,
not implemented by default, rather than shipping a comparison that looks rigorous but isn't.

In [ ]:
frozen_primary_path = config.V2_FINAL_TABLES_DIR / "final_aipw_overall_table.csv"
frozen_primary = pd.read_csv(frozen_primary_path).iloc[0]

comparison = pd.DataFrame([
    {"cohort": "full_cohort_primary (Notebook 07)", "n": int(frozen_primary["n_analytic"]),
     "risk_difference_pct": frozen_primary["risk_difference_pct"],
     "ci_low_pct": frozen_primary["risk_difference_ci_low_pct"], "ci_high_pct": frozen_primary["risk_difference_ci_high_pct"],
     "risk_ratio": frozen_primary["risk_ratio"],
     "rr_ci_low": frozen_primary["risk_ratio_ci_low"], "rr_ci_high": frozen_primary["risk_ratio_ci_high"]},
    {"cohort": "first_birth_only (this notebook)", "n": first_birth_result["n"],
     "risk_difference_pct": first_birth_result["risk_difference"] * 100,
     "ci_low_pct": first_birth_result["rd_ci"][0] * 100, "ci_high_pct": first_birth_result["rd_ci"][1] * 100,
     "risk_ratio": first_birth_result["risk_ratio"],
     "rr_ci_low": first_birth_result["rr_ci"][0], "rr_ci_high": first_birth_result["rr_ci"][1]},
])

ci_overlap = not (
    comparison.loc[1, "ci_high_pct"] < comparison.loc[0, "ci_low_pct"] or
    comparison.loc[0, "ci_high_pct"] < comparison.loc[1, "ci_low_pct"]
)
print(comparison)
print(f"\nConfidence intervals overlap: {ci_overlap} "
      f"({'no strong evidence the two estimates differ' if ci_overlap else 'the two estimates are distinguishable at the 95% level'})")

## 8. Figure -- comparison forest plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
y_pos = [0, 1]
ax.errorbar(
    comparison["risk_difference_pct"], y_pos,
    xerr=[comparison["risk_difference_pct"] - comparison["ci_low_pct"], comparison["ci_high_pct"] - comparison["risk_difference_pct"]],
    fmt="o", color="steelblue", ecolor="gray", capsize=4,
)
ax.set_yticks(y_pos)
ax.set_yticklabels(comparison["cohort"])
ax.invert_yaxis()
ax.axvline(0, color="black", linestyle=":", linewidth=1)
ax.set_xlabel("Adjusted risk difference, private minus public (percentage points)")
ax.set_title("Full-Cohort Primary vs. First-Birth-Only Adjusted Sector Gap")
fig.tight_layout()
fig.savefig(OUTPUTS_DIR / "first_birth_comparison_plot.png", dpi=200)
plt.show()
print("Saved:", OUTPUTS_DIR / "first_birth_comparison_plot.png")

## 9. QA / interpretation checks (explicit)

In [ ]:
print("First-birth definition verified: birth_order == 1 (not birth_index).")
print(f"Records excluded for missing age_at_first_birth (not substituted with maternal_age): {n_missing_age}")
print("birth_order confirmed constant within this cohort and excluded as a predictor (Section 3 assertion).")
print()
print("No prior-Cesarean confounding by construction: a first birth cannot have a prior Cesarean, so this "
      "specific unmeasured confounder (the focus of Extension 1's bias analysis) cannot operate within this "
      "cohort. This does NOT mean all unmeasured confounding is removed -- other first-birth-specific obstetric "
      "indications (fetal compromise, labor dystocia, cephalopelvic disproportion, etc.) remain unmeasured in "
      "NFHS-5 and are not addressed here.")
print()
print("This notebook does not claim to be the first low-risk/nulliparous facility-sector analysis in India -- "
      "the restriction itself is a standard epidemiological design choice, not a novel contribution.")

## 10. Save metadata

In [ ]:
metadata = {
    "random_state": RANDOM_STATE,
    "cohort": {
        "n_first_birth_total": int(len(first_birth)),
        "n_excluded_missing_age_at_first_birth": int(n_missing_age),
        "n_first_birth_modeling": int(len(first_birth_modeling)),
    },
    "confounder_set": FIRST_BIRTH_CONFOUNDERS,
    "birth_order_excluded_reason": "Constant (== 1) within this cohort; would be a degenerate predictor.",
    "age_at_first_birth_included_reason": (
        "Exact age at the analyzed birth by construction for birth_order==1 records -- unlike maternal_age, "
        "which is age at interview and not verified as age at the analyzed birth (see notebooks/v2/06)."
    ),
    "nuisance_models": "identical architecture to notebooks/v2/07, refit independently on the first-birth cohort",
    "comparison_to_primary": {
        "method": "side-by-side point estimates and CIs, not a formal joint difference CI",
        "reason": (
            "First-birth women are a subset of, not independent from, the full analytic cohort; naively pairing "
            "bootstrap replicates (as Extension 3 does for the genuinely independent NFHS-4/NFHS-5 samples) would "
            "understate their true correlation and produce an invalid interval."
        ),
        "full_cohort_rd_pct": float(frozen_primary["risk_difference_pct"]),
        "first_birth_rd_pct": float(first_birth_result["risk_difference"] * 100),
        "ci_overlap": bool(ci_overlap),
    },
    "limitations": [
        "Removes prior-Cesarean confounding by construction only -- other first-birth-specific unmeasured "
        "obstetric indications (fetal compromise, labor dystocia) are not addressed.",
        "Not claimed as a novel first-of-its-kind low-risk/nulliparous sector analysis in India.",
    ],
}

with open(OUTPUTS_DIR / "first_birth_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print("Saved:", OUTPUTS_DIR / "first_birth_metadata.json")

## 11. Documentation -- key design decisions and why

**Why `age_at_first_birth` replaces `birth_order` rather than just dropping a confounder.**
Simply removing `birth_order` without replacement would shrink the confounder set for no methodological reason.
`age_at_first_birth` is available, exact for this specific subgroup (unlike `maternal_age` project-wide), and a
plausible confounder (younger/older first-time mothers may differ in both facility choice and Cesarean risk) --
so it's a like-for-like, well-justified substitution rather than an arbitrary swap.

**Why nuisance models are refit from scratch on the first-birth cohort, not reused from Notebook 07.**
The primary model's nuisance models were trained on the *full* cohort with `birth_order` as a live predictor
(it varies from 1 to 16+ there). Applying that model to a subset where `birth_order` is constant would not
reflect a model that ever learned anything meaningful about this specific subpopulation's other confounders'
relationships -- an honest first-birth-specific estimate needs its own fit, exactly as the handoff specifies.

**Why the comparison to the full-cohort primary is deliberately not a formal difference test.**
See Section 7's markdown -- the two cohorts overlap (first births are a subset of the full cohort), so a
correlation-naive method for combining their uncertainty would be actively misleading rather than merely
imprecise. Reporting both estimates with their own honestly-computed CIs, and stating plainly whether they
visually overlap, is the more defensible choice here.

**What this notebook does and does not establish.**
It shows whether the adjusted sector gap persists among first-time mothers specifically -- a subgroup where
prior-Cesarean confounding is structurally impossible. It does not establish that *all* unmeasured confounding
is absent in this subgroup, and it does not claim any novelty for the restriction itself.